# Análise Exploratória - ESA Anomaly Dataset

Este notebook faz parte de uma Iniciação Científica que estuda como avaliar a **confiabilidade de alertas em telemetria espacial**.

## Contexto

Satélites geram continuamente séries temporais chamadas de **telemetria** — medidas de temperatura, tensão, corrente e outros parâmetros dos seus subsistemas. Sistemas de monitoramento observam esses dados e emitem **alertas** quando detectam  comportamentos fora do padrão.

O problema central é que **um alerta não significa necessariamente uma falha real**.  O mesmo comportamento atípico pode ser:

- 🔴 Uma **anomalia real** — algo efetivamente errado no satélite
- 🟡 Um **evento nominal raro** — uma operação planejada, incomum mas esperada  (ex: uma manobra, uma calibração)
- ⚪ **Ruído ou problema de qualidade dos dados**

Esta análise exploratória vai nos ajudar a **entender a estrutura dos dados disponíveis** antes de construir qualquer detector ou avaliador de alertas.

In [47]:
# import das bibliotecas e carregamento dos dataset
import pickle
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../../data/ESA/ESA-Mission1")

def load_zip_pickle(path: Path) -> pd.DataFrame:
    """Carrega um DataFrame pickled dentro de um ZIP."""
    with zipfile.ZipFile(path) as z:
        name = z.namelist()[0]
        return pickle.loads(z.read(name))

## O que vamos fazer aqui

Este notebook segue uma progressão natural:

1. **Carregar e inspecionar os metadados** (CSVs) — entender o que existe antes de abrir os arquivos grandes
2. **Carregar as séries temporais** dos canais 41 - 46 — ver como os dados reais se parecem
3. **Visualizar os canais** — identificar padrões, anomalias e eventos nominais raros visualmente
4. **Carregar e visualizar telecomandos** — entender quando e com que frequência comandos foram enviados
5. **Explorar a possibilidade de combinação** — verificar se faz sentido cruzar séries de 
   telemetria com o histórico de telecomandos (join temporal)

## Escopo desta etapa: Mission1

Nesta primeira análise exploratória, vamos focar na **Mission1** do ESA-ADB. A análise equivalente da Mission2 fica para uma etapa posterior, depois de compreendermos e documentarmos este recorte.

### Recorte atual

- **Telemetria:** canais 41–46, que são os canais disponíveis neste subconjunto local.
- **Eventos:** rótulos associados aos canais 41–46, cruzados com `anomaly_types.csv`. As contagens de eventos desta análise serão de **IDs únicos presentes no recorte**, não do total da Mission1.
- **Telecomandos:** catálogo da Mission1 e históricos disponíveis em `telecommands/`, usados para caracterização e análise de proximidade temporal. Uma associação temporal não demonstra causalidade.

### O que esta EDA deve produzir

Vamos caracterizar os eventos, os canais, a amostragem, a qualidade dos dados e o histórico de telecomandos. Também podemos construir uma **baseline exploratória** para visualizar como regras simples se comportam; ela não substitui a avaliação formal do detector, que deverá respeitar os períodos de treino, validação e teste e as métricas apropriadas do benchmark.

As conclusões desta etapa ficam limitadas à Mission1 e aos canais selecionados. Não serão generalizadas para todos os canais da missão nem para a Mission2.

## O dataset: ESA-ADB Mission1

Usamos o **ESA Anomaly Detection Benchmark (ESA-ADB)**, um benchmark público  publicado pela Agência Espacial Europeia (ESA) em parceria com a KP Labs.

> 📎 [Artigo original](https://arxiv.org/abs/2406.17826) 

> [Repositório oficial](https://github.com/kplabs-pl/ESA-ADB) 

> [Zenodo (dados)](https://doi.org/10.5281/zenodo.12528696)

### ⚠️ Importante: os dados são anonimizados

Os nomes reais dos canais, subsistemas e telecomandos **não são divulgados**.  Por isso, você verá identificadores genéricos como `channel_41`, `subsystem_5`  e `telecommand_198`. Isso é intencional — protege informações proprietárias da missão.

### O que está disponível neste subconjunto

O ESA-ADB é grande; aqui trabalhamos com um **subconjunto leve** da Mission1:

| Arquivo | O que contém |
|---|---|
| `channels.csv` | Catálogo dos 76 canais: subsistema, unidade física, grupo, flag de alvo |
| `labels.csv` | Eventos rotulados por canal (início, fim, tipo de evento) |
| `anomaly_types.csv` | Tipos de eventos: Anomalia vs. Evento Nominal Raro |
| `telecommands.csv` | Catálogo dos 698 tipos de telecomandos com prioridade |
| `channels/channel_4{1..6}.zip` | Séries temporais dos canais 41 a 46 |
| `telecommands/telecommand_*.zip` | Histórico de quando cada comando foi executado |

### Estrutura dos arquivos de série temporal

Os arquivos `.zip` de canais e telecomandos contêm **DataFrames pickled**  (serializados em formato Python). Cada um tem:

- **Índice:** `datetime` — o timestamp da medição ou execução do comando
- **Coluna:** o valor medido (canal) ou uma flag `1` indicando execução (telecomando)

Os **6 canais disponíveis** (41-46) pertencem todos ao mesmo subsistema e grupo,  o que significa que monitoram aspectos relacionados do satélite.

Os **telecomandos** registram os comandos enviados ao satélite ao longo dos anos. Cada arquivo corresponde a um tipo de comando e armazena todos os momentos em que ele foi executado. Alguns comandos são raros (poucas dezenas de execuções) e outros são frequentes (centenas de milhares).


In [48]:
channels_meta = pd.read_csv(DATA_DIR / "channels.csv")
labels = pd.read_csv(DATA_DIR / "labels.csv")
anomaly_types = pd.read_csv(DATA_DIR / "anomaly_types.csv")
tc_meta = pd.read_csv(DATA_DIR / "telecommands.csv")

### 📡 Tabela 1: `channels.csv` — "Quais sensores existem no satélite?"

Cada linha representa um **canal de telemetria** — uma série temporal coletada  continuamente pelo satélite.

> Exemplo: Imagine que o satélite é como um avião. Dentro dele existem dezenas de sensores  medindo coisas o tempo todo: temperatura do motor, nível de combustível, pressão  da cabine, etc. No mundo dos satélites, cada um desses sensores é chamado de **canal**. Esta tabela é basicamente o **manual de bordo** — ela lista todos os canais  existentes e diz o que cada um mede.

> 💡 Dos 76 canais, 58 são `Target=YES`. Os 6 canais que temos disponíveis (41–46) são todos do mesmo subsistema e grupo, ou seja, medem grandezas relacionadas entre si.

| Coluna | O que significa |
|---|---|
| `Channel` | Nome identificador do canal (ex: `channel_41`). Anonimizado. |
| `Subsystem` | Subsistema do satélite ao qual o canal pertence. Um subsistema agrupa componentes relacionados (ex: energia, atitude, comunicação). Também anonimizado. |
| `Physical Unit` | Tipo de grandeza física medida (ex: temperatura, tensão, corrente). Anonimizado. |
| `Group` | Agrupamento temático dos canais — canais do mesmo grupo tendem a medir aspectos relacionados. |
| `Target` | `YES` indica que este canal é **alvo da detecção de anomalias**. `NO` significa que o canal existe apenas como contexto e não é avaliado diretamente. |


In [4]:
print("=== channels_meta ===")
display(channels_meta.head())

print(channels_meta['Target'].value_counts())
print('')

print("=== channels_meta (canais 41–46) ===")
canais_disponiveis = [f"channel_{i}" for i in range(41, 47)]
display(channels_meta[channels_meta["Channel"].isin(canais_disponiveis)])

=== channels_meta ===


,Channel,Subsystem,Physical Unit,Group,Target
0,channel_1,subsystem_1,physical_unit_1,1,NO
1,channel_2,subsystem_1,physical_unit_1,1,NO
2,channel_3,subsystem_1,physical_unit_1,1,NO
3,channel_4,subsystem_1,physical_unit_2,2,NO
4,channel_5,subsystem_1,physical_unit_2,2,NO


Target
YES    58
NO     18
Name: count, dtype: int64

=== channels_meta (canais 41–46) ===


,Channel,Subsystem,Physical Unit,Group,Target
40,channel_41,subsystem_5,physical_unit_4,8,YES
41,channel_42,subsystem_5,physical_unit_4,8,YES
42,channel_43,subsystem_5,physical_unit_4,8,YES
43,channel_44,subsystem_5,physical_unit_4,8,YES
44,channel_45,subsystem_5,physical_unit_4,8,YES
45,channel_46,subsystem_5,physical_unit_4,8,YES


### 🏷️ Tabela 2: `labels.csv` — "O que realmente aconteceu, e quando?"

Quando engenheiros da ESA analisaram os dados históricos do satélite, eles identificaram eventos importantes e anotaram: *"entre essa data e essa data, aconteceu alguma coisa nesse canal"*.

Essa tabela é o resultado dessas anotações — o nosso **gabarito oficial**. Ela nos diz o que de fato ocorreu, o que é essencial para avaliar se um detector automático teria acertado ou errado.

| Coluna | Significado simples |
|---|---|
| `ID` | O "nome" do evento. Se o mesmo `ID` aparece em várias linhas (como `id_1` nas 5 primeiras linhas), significa que aquele evento afetou vários canais ao mesmo tempo — não foi só um sensor que reagiu, foram vários juntos. |
| `Channel` | Qual sensor registrou esse evento. |
| `StartTime` | Quando o evento começou. |
| `EndTime` | Quando o evento terminou. |

> ⚠️ **Repare numa coisa:** o `ID` aqui identifica a *ocorrência* — ou seja,  "aconteceu uma vez". Ele **não diz** se foi uma coisa grave ou inofensiva. Para saber isso, precisamos cruzar com a próxima tabela.

> Também podemos notar que algumas linhas têm valores estranhos no `StartTime` (aparece `"Resultado"` em vez de uma data). Isso é um dado ausente ou corrompido — algo que vamos tratar na hora de usar os dados de verdade.


In [5]:
print("=== labels ===")
display(labels.head())

print("=== labels (canais 41–46) ===")
canais_disponiveis = [f"channel_{i}" for i in range(41, 47)]
display(labels[labels["Channel"].isin(canais_disponiveis)])

=== labels ===


,ID,Channel,StartTime,EndTime
0,id_1,channel_12,2004-12-01T20:42:15.429Z,2004-12-08 22:55:45.429000+00:00
1,id_1,channel_13,2004-12-01T20:42:15.429Z,2004-12-08 22:55:45.429000+00:00
2,id_1,channel_14,2004-12-01T20:43:45.429Z,2004-12-02 02:57:15.429000+00:00
3,id_1,channel_15,Resultado,2004-12-02 02:58:45.429000+00:00
4,id_1,channel_16,2004-12-01T20:43:45.429Z,2004-12-16 16:52:30.429000+00:00


=== labels (canais 41–46) ===


,ID,Channel,StartTime,EndTime
29,id_1,channel_41,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
30,id_1,channel_42,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
31,id_1,channel_43,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
32,id_1,channel_44,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
33,id_1,channel_45,2004-12-02T03:07:17.646Z,2004-12-08 23:35:55.146000+00:00
...,...,...,...,...
3574,id_200,channel_42,2000-10-10T01:09:58.323Z,2000-10-11 00:11:58.323000+00:00
3575,id_200,channel_43,2000-10-10T00:23:28.323Z,2000-10-11 04:46:58.323000+00:00
3576,id_200,channel_44,2000-10-10T00:40:58.323Z,2000-10-11 03:24:58.323000+00:00
3577,id_200,channel_45,2000-10-10T00:46:58.323Z,2000-10-11 01:15:58.323000+00:00


### 🔍 Tabela 3: `anomaly_types.csv` — "Esse evento foi um problema ou era esperado?"

Nem todo comportamento fora do normal é uma falha. Imagine um avião realizando  uma manobra de pouso: os sensores vão registrar coisas bem diferentes do voo normal — mas isso é *esperado*, não é um problema.

Esta tabela classifica cada tipo de evento em duas categorias principais:

| Coluna | Significado simples |
|---|---|
| `ID` | Liga com o `ID` da tabela anterior — é assim que descobrimos *o que era* aquele evento. |
| `Class` / `Subclass` | Uma classificação técnica mais detalhada do tipo de evento. Os nomes reais foram anonimizados, mas eventos da mesma classe têm características em comum. |
| `Category` | **A coluna mais importante de todas.** Só tem dois valores possíveis: |
| | 🔴 **`Anomaly`** → algo deu errado. É o tipo de evento que um sistema de monitoramento *precisa* detectar. |
| | 🟡 **`Rare Event`** → aconteceu algo incomum, mas que era *esperado* — uma manobra, uma calibração, uma mudança de modo de operação. Não é uma falha, mas parece estranho nos dados. |
| `Locality` | Se o evento foi visível em um trecho pequeno (`Local`) ou em um período longo (`Global`) da série de dados. |
| `Length` | Aqui todos os eventos são `Subsequence` — ou seja, têm começo e fim definidos, não são pontos isolados. |

> 💡 **Por que isso importa para a pesquisa?**  
> Um detector automático simples vai gerar um alerta tanto para uma `Anomaly` quanto para um `Rare Event` — os dois parecem "estranhos" nos dados. O nosso objetivo é entender se é possível usar informações de contexto (como comandos  enviados ao satélite) para ajudar a distinguir um do outro, sem precisar de um especialista humano.


In [6]:
print("=== anomaly_types ===")
display(anomaly_types.head())

=== anomaly_types ===


,ID,Class,Subclass,Category,Dimensionality,Locality,Length
0,id_1,class_6,subclass_1,Rare Event,Multivariate,Global,Subsequence
1,id_2,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence
2,id_3,class_7,subclass_1,Anomaly,Multivariate,Global,Subsequence
3,id_4,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence
4,id_5,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence


### 🛰️ Tabela 4: `tc_meta` (`telecommands.csv`) — "Que comandos podem ser enviados ao satélite?"

O satélite não funciona sozinho. A equipe em terra pode enviar **comandos** para ele — instruções como "ligue esse sensor", "mude a orientação", "entre em modo de economia de energia". Esses comandos se chamam **telecomandos** (TC).

Esta tabela é os **tipos de comandos disponíveis**: lista todos os 698 tipos de telecomandos que existem nessa missão.

| Coluna | Significado simples |
|---|---|
| `Telecommand` | O código do tipo de comando (ex: `telecommand_198`). Anonimizado, assim como os canais. |
| `Priority` | Uma indicação de quão "urgente" ou "importante" esse tipo de comando é. |

> 📌 **O que essa tabela NÃO tem:** ela não diz *quando* cada comando foi usado. 

> O **histórico real de execuções** — ou seja, *"em qual data e hora o da `telecommand_198` foi enviado ao satélite?"* — está nos arquivos `.zip` dentro da pasta `telecommands/`. Vamos carregar esses arquivos nas próximas células para explorar isso.

> A ideia é verificar: se um telecomando foi enviado pouco antes de um sensor registrar algo incomum, isso pode ser uma pista de que o "comportamento estranho" era na verdade uma resposta *esperada* ao comando — e não uma falha.


In [7]:
print("=== tc_meta ===")
display(tc_meta.head(-10))

=== tc_meta ===


,Telecommand,Priority
0,telecommand_1,1
1,telecommand_2,1
2,telecommand_3,1
3,telecommand_4,1
4,telecommand_5,1
...,...,...
683,telecommand_684,1
684,telecommand_685,1
685,telecommand_686,1
686,telecommand_687,1



## 📂 Os dados reais: séries temporais dos canais 41–46

Até agora vimos apenas as **tabelas de metadados** — informações *sobre* os dados. Agora vamos abrir os arquivos que contêm as **leituras reais dos sensores**: o que o satélite mediu, segundo a segundo, ao longo de anos de operação.

### Como esses arquivos funcionam?

Cada canal tem um arquivo `.zip` próprio. Dentro de cada ZIP há um único arquivo no formato **pickle** — um formato do Python para salvar objetos diretamente, sem precisar converter para texto. O resultado, ao carregar, é um `DataFrame` do pandas com esta estrutura:

| O quê | Detalhes |
|---|---|
| **Índice** | Timestamps — data e hora de cada leitura, com precisão de milissegundos |
| **Coluna** | O valor medido pelo sensor naquele instante (número decimal) |

> 💡 Os timestamps **não são igualmente espaçados** — o satélite não mede sempre no mesmo intervalo. Às vezes há leituras com segundos de diferença, às vezes com minutos. Isso é importante de entender antes de fazer qualquer  análise temporal.

> ⚠️ Cada arquivo é grande: o `channel_41`, por exemplo, tem mais de  *15 milhões de linhas** — representando anos de leituras contínuas. Por isso, para visualizar, vamos usar amostras.


In [9]:
# Carregar todos os canais disponíveis
channel_ids = [41, 42, 43, 44, 45, 46]
channels = {}

for cid in channel_ids:
    path = DATA_DIR / "channels" / f"channel_{cid}.zip"
    channels[f"channel_{cid}"] = load_zip_pickle(path)

# Inspecionar o primeiro como exemplo
ch41 = channels["channel_41"]

print("=== Primeiras leituras ===")
display(ch41.head())

print("=== Últimas leituras ===")
display(ch41.tail())

print("=== Resumo estatístico ===")
display(ch41.describe())

print("\n=== Informações gerais de todos os canais ===")
for nome, df in channels.items():
    inicio = df.index.min()
    fim = df.index.max()
    duracao = fim - inicio
    print(f"{nome}: {len(df):>12,.0f} leituras | {inicio.date()} → {fim.date()} | duração: {duracao.days} dias")


/tmp/ipykernel_76631/3751775339.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  return pickle.loads(z.read(name))


=== Primeiras leituras ===


,channel_41
datetime,
2000-01-01 00:00:16.353,0.812578
2000-01-01 00:00:46.353,0.821213
2000-01-01 00:01:08.853,0.808654
2000-01-01 00:01:16.353,0.819642
2000-01-01 00:01:46.353,0.821996


=== Últimas leituras ===


,channel_41
datetime,
2013-12-31 23:57:39.657,0.806299
2013-12-31 23:58:09.657,0.806299
2013-12-31 23:58:39.657,0.806299
2013-12-31 23:59:09.657,0.806299
2013-12-31 23:59:39.657,0.808654


=== Resumo estatístico ===


,channel_41
count,1.538117e+07
mean,8.096232e-01
std,1.565005e-02
min,0.000000e+00
25%,8.055143e-01
50%,8.094388e-01
75%,8.141486e-01
max,9.821104e-01



=== Informações gerais de todos os canais ===
channel_41:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_42:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_43:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_44:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_45:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias
channel_46:   15,381,169 leituras | 2000-01-01 → 2013-12-31 | duração: 5113 dias


### Interpretando os Dados Acima

#### A coluna `channel_4*` — o que são esses números?

A coluna recebe o nome do canal e armazena o **valor medido pelo sensor** naquele instante. Como os dados são anonimizados, não sabemos a grandeza física real (temperatura? tensão? pressão?) nem a unidade de medida.

O que importa é analisarmos  **como ele muda ao longo do tempo**: uma leitura muito diferente do padrão histórico é o que nos 
interessa detectar.


#### Primeiras leituras — início da missão

O sensor começou a ser monitorado em **1º de janeiro de 2000**. Nas primeiras  eituras, os valores ficam em torno de **0.81–0.82** e o intervalo entre medições varia: ora 30 segundos, ora 22 segundos, ora 7 segundos. Isso confirma que a frequência de amostragem **não é fixa** — o satélite não mede sempre no mesmo ritmo.


#### Resumo estatístico

| O que olhar | O que significa neste canal |
|---|---|
| **Média ≈ 0.810** | O valor "normal" que o sensor costuma registrar |
| **Desvio padrão ≈ 0.016** | O sensor é bem estável — a variação típica é de apenas ~2% em torno da média |
| **Mínimo = 0.000** | ⚠️ Pelo menos uma vez o sensor registrou zero — muito abaixo do padrão. Pode ser uma anomalia, sensor desligado ou dado corrompido |
| **Máximo ≈ 0.982** | O maior valor já registrado — ainda dentro de um range razoável, mas ~21% acima da média |
| **Percentis 25–75%: 0.805 a 0.814** | Em 50% do tempo, o sensor fica numa faixa estreita de apenas 0.009 de amplitude — confirma a estabilidade |


#### Todos os canais têm exatamente a mesma quantidade de leituras

Os 6 canais têm **15.381.169 leituras cada**, do mesmo período (2000–2013). Isso não é coincidência — como pertencem ao mesmo subsistema e grupo, são coletados de forma sincronizada. 

### 🛰️ Como os arquivos de telecomandos são organizados?


A pasta `telecommands/` tem um arquivo `.zip` para **cada tipo de telecomando** — ou seja, 698 arquivos no total.

> Imagine que cada arquivo é como um **diário de bordo de um botão específico do painel de controle**: ele registra todas as vezes que aquele botão foi pressionado, com a data e hora exata.

Cada arquivo `.zip` contém um DataFrame pickled com a seguinte estrutura:

| O quê | Detalhes |
|---|---|
| **Índice** | `datetime` — o timestamp de quando o comando foi executado |
| **Coluna** | O valor `1` em todas as linhas — indica simplesmente que o comando *ocorreu* naquele instante |

> **Atenção:** ao contrário dos canais (que armazenam valores numéricos contínuos), os arquivos de telecomandos não medem grandezas — eles registram *eventos*. O valor `1` é apenas uma marcação: "sim, isso aconteceu".

> 💡 **Por que isso importa?** - Hipótese: Se um telecomando foi enviado pouco antes de um sensor registrar algo estranho, isso pode indicar que o "comportamento estranho" era na verdade uma *resposta esperada* ao comando — e não uma falha real. 

In [10]:
# Carregar alguns telecomandos como exemplo
# Usamos o tc_198 (maior arquivo = mais execuções) e alguns pequenos para comparar
tcs_para_explorar = [198, 270, 482, 10]

tc_series = {}
for tc_id in tcs_para_explorar:
    path = DATA_DIR / "telecommands" / f"telecommand_{tc_id}.zip"
    tc_series[f"telecommand_{tc_id}"] = load_zip_pickle(path)

# Inspecionar o telecommand_198 como exemplo (o mais frequente)
tc198 = tc_series["telecommand_198"]

print("=== Primeiras execuções do telecommand_198 ===")
display(tc198.head())

print("=== Últimas execuções ===")
display(tc198.tail())

print("\n=== Informações gerais dos telecomandos carregados ===")
for nome, df in tc_series.items():
    inicio = df.index.min()
    fim = df.index.max()
    duracao = fim - inicio
    print(f"{nome}: {len(df):>10,.0f} execuções | {inicio.date()} → {fim.date()} | duração: {duracao.days} dias")


=== Primeiras execuções do telecommand_198 ===


/tmp/ipykernel_76631/3751775339.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  return pickle.loads(z.read(name))


,telecommand_198
2000-01-03 07:29:27.549,1
2000-01-03 07:29:29.337,1
2000-01-03 07:29:30.993,1
2000-01-03 07:29:33.024,1
2000-01-03 07:29:34.824,1


=== Últimas execuções ===


,telecommand_198
2013-12-31 16:21:06.300,1
2013-12-31 16:21:08.106,1
2013-12-31 16:21:09.915,1
2013-12-31 16:21:11.742,1
2013-12-31 16:21:13.515,1



=== Informações gerais dos telecomandos carregados ===
telecommand_198:    266,724 execuções | 2000-01-03 → 2013-12-31 | duração: 5111 dias
telecommand_270:    327,502 execuções | 2000-01-03 → 2013-12-31 | duração: 5111 dias
telecommand_482:    519,992 execuções | 2000-02-26 → 2013-04-25 | duração: 4807 dias
telecommand_10:        897 execuções | 2000-02-10 → 2013-09-12 | duração: 4962 dias


### Interpretando os dados acima

#### O que a diferença de tamanho dos arquivos nos diz?

| Telecomando | Tamanho aprox. | O que isso sugere |
|---|---|---|
| `telecommand_198` | ~1.2 MB | Executado com muita frequência |
| `telecommand_270` | ~1.4 MB | Também muito frequente |
| `telecommand_10` | ~5 KB | Raro — executado poucas vezes ao longo de anos |

> ⚠️ **Cuidado metodológico importante:** saber que um telecomando foi executado **não prova** que ele causou uma mudança no sensor. Os dados são anonimizados — não sabemos o que cada comando faz fisicamente. O que podemos observar é **proximidade temporal**, não causalidade.


## Preparação e auditoria dos rótulos

Esta seção verifica e organiza a tabela de rótulos (`labels`) e a liga à tabela de tipos de evento (`anomaly_types`). O resultado é a tabela final `eventos_unicos`, com **uma linha por evento**, que será a base das análises seguintes.

Os rótulos foram o primeiro alvo da auditoria porque são a única fonte que informa **quando** cada evento começou e terminou. As análises seguintes usam essas datas para recortar as séries de telemetria e o histórico de telecomandos. Um erro nos rótulos afetaria todas essas etapas.

Esta seção **não** audita as séries dos canais nem os arquivos de telecomandos. Essas verificações serão feitas no início das seções que usam esses dados (seção 2 para as séries e seção 3 para os telecomandos).

Nenhum arquivo original é alterado. As linhas com problema não são apagadas: são separadas em uma tabela própria, para que possam ser consultadas depois.

### Como as tabelas se ligam

Os dados estão em tabelas diferentes, e cada ligação usa uma chave em comum:

| Ligação | Chave | O que permite saber |
|---|---|---|
| `labels` ↔ `channels.csv` | `Channel` (ex.: `channel_41`) | Subsistema, grupo e se o canal é alvo da detecção. |
| `labels` ↔ `anomaly_types` | `ID` (ex.: `id_100`) | Categoria (anomalia, evento raro ou lacuna de comunicação) e classe do evento. **É a ligação feita nesta seção.** |
| `labels` ↔ série do canal | `Channel` + tempo | O nome do canal indica qual série abrir (`channel_41.zip`), e o início e o fim do evento indicam qual trecho recortar. |
| `labels` ↔ telecomandos | tempo | O início do evento define uma janela de tempo, e a pergunta é quais telecomandos foram executados nela. |
| `telecommands.csv` ↔ `telecommand_*.zip` | `Telecommand` (ex.: `telecommand_198`) | Liga a prioridade ao arquivo com as execuções. |

Em resumo: a tabela `labels` indica **qual canal** (nome) e **em qual período** (datas) cada evento ocorreu, e é por ela que as outras tabelas se conectam.

Os telecomandos **não têm ligação direta com os canais**. Os nomes são anonimizados, então não é possível saber qual comando age sobre qual canal. A ligação é feita apenas pela **proximidade no tempo**, o que permite observar associação, mas não provar causa.

### Como a tabela final é criada

| Passo | O que é feito | Linhas após o passo |
|---|---|---|
| 1 | Ler `labels.csv` | 3.589 |
| 2 | Copiar a tabela (`labels_audit`) e converter as datas para colunas novas (`StartTimeUTC`, `EndTimeUTC`) | 3.589 |
| 3 | Filtrar os canais 41 a 46 (`labels_41_46_raw`) | 1.255 |
| 4 | Separar as linhas inválidas (`labels_invalidas_41_46`) e manter as válidas (`labels_clean`) | 1.255 válidas, 0 inválidas |
| 5 | Verificar se há IDs repetidos em `anomaly_types` e unir as duas tabelas pelo `ID` (`labels_rotuladas`) | 1.255 |
| 6 | Verificar se todos os IDs encontraram um tipo (coluna `_merge`) | 1.255 (0 sem correspondência) |
| 7 | Agrupar por `ID` e criar a tabela `eventos_unicos` | **118 eventos** |

No passo 7, cada grupo de linhas com o mesmo `ID` vira uma única linha, com as colunas:
- `inicio_utc`: o menor horário de início entre todos os canais do evento;
- `fim_utc`: o maior horário de fim entre todos os canais do evento;
- `canais_afetados`: quantos canais diferentes registraram o evento;
- `n_intervalos_anotados`: quantas linhas de `labels` o evento tinha;
- `Category`, `Class` e `Subclass`: a classificação do evento;
- `duracao_envelope`: a diferença entre `fim_utc` e `inicio_utc`.

**Por que agrupar por `ID`?** Cada linha de `labels` é um intervalo anotado em **um canal**. Um mesmo evento aparece em vários canais e, às vezes, em mais de um trecho por canal. Por exemplo, o evento `id_100` tem 12 linhas: 6 canais, com 2 trechos em cada um. Contar linhas daria um número de eventos maior que o real.

### Decisões tomadas e por quê

**1. Trabalhar em uma cópia, com colunas novas para as datas.** Os valores originais ficam preservados e podem ser comparados com os convertidos.

**2. Converter as datas com `errors="coerce"`.** Um valor que não pode ser lido como data (como o texto "Resultado" encontrado no início do notebook) vira um valor vazio (`NaT`). O código não é interrompido, e o problema pode ser contado e localizado.

**3. Converter as datas para UTC (`utc=True`).** As colunas `StartTime` e `EndTime` estavam em formatos diferentes. Com o mesmo fuso, as datas podem ser comparadas e subtraídas sem erro. As séries de telemetria não têm fuso horário. Ao cruzá-las com os rótulos, será preciso remover o fuso das colunas de rótulos.

**4. Limitar a análise aos canais 41 a 46.** São os canais disponíveis para esta pesquisa. A tabela de resumo também informa os problemas do arquivo inteiro, para dar contexto.

**5. Definir linha inválida.** Uma linha é considerada inválida se tiver pelo menos um destes problemas: data de início ou de fim ausente ou ilegível; data de fim anterior à de início; ID ausente. Cada problema impede uma análise: sem data, o evento não pode ser posicionado no tempo; com fim anterior ao início, a duração fica negativa; sem ID, não se sabe a qual evento a linha pertence.

**6. Separar as linhas inválidas em vez de excluí-las.** O que fica de fora da análise fica registrado e pode ser justificado.

**7. Verificar a união com `anomaly_types`.** Antes de unir, o código confere se há IDs repetidos em `anomaly_types` e para com um erro se houver. A união usa três opções:
- `how="left"`: mantém todas as linhas de `labels_clean`, mesmo que algum ID não tenha tipo correspondente;
- `validate="many_to_one"`: confirma que cada ID aparece uma única vez em `anomaly_types`, o que evita linhas duplicadas sem aviso;
- `indicator=True`: cria a coluna `_merge`, que mostra quais linhas encontraram correspondência.

**8. Usar `first` para `Category`, `Class` e `Subclass`.** Isso é seguro porque cada ID tem uma única linha em `anomaly_types` (garantido pela verificação anterior).

### Cuidados ao interpretar `duracao_envelope`

A coluna `duracao_envelope` mede o **tempo total entre o primeiro início e o último fim**. Ela não é a duração exata do evento, por dois motivos:
- Quando um evento tem trechos separados, a janela inclui também os períodos sem anotação entre eles.
- Segundo os autores do dataset, o horário de **fim** dos eventos foi anotado com menos precisão do que o de início.

### Colunas que não foram usadas

As colunas `Dimensionality`, `Locality` e `Length` de `anomaly_types.csv` não foram trazidas para `eventos_unicos`. Os autores as calcularam considerando todos os canais da missão, e não apenas os canais 41 a 46. Por isso, elas não descrevem corretamente o recorte desta pesquisa.

In [53]:
canais_disponiveis = [f"channel_{i}" for i in range(41, 47)]

# Preserve labels as read from CSV and normalize timestamps in separate columns.
labels_audit = labels.copy()
labels_audit["StartTimeUTC"] = pd.to_datetime(
    labels_audit["StartTime"], errors="coerce", utc=True
)
labels_audit["EndTimeUTC"] = pd.to_datetime(
    labels_audit["EndTime"], errors="coerce", utc=True
)

labels_41_46_raw = labels_audit.loc[
    labels_audit["Channel"].isin(canais_disponiveis)
].copy()

invalid_date = (
    labels_41_46_raw["StartTimeUTC"].isna()
    | labels_41_46_raw["EndTimeUTC"].isna()
)
invalid_order = (
    labels_41_46_raw["StartTimeUTC"].notna()
    & labels_41_46_raw["EndTimeUTC"].notna()
    & (labels_41_46_raw["EndTimeUTC"] < labels_41_46_raw["StartTimeUTC"])
)
missing_id = labels_41_46_raw["ID"].isna()
invalid_rows = invalid_date | invalid_order | missing_id

labels_invalidas_41_46 = labels_41_46_raw.loc[invalid_rows].copy()
labels_clean = labels_41_46_raw.loc[~invalid_rows].copy()

if anomaly_types["ID"].duplicated().any():
    raise ValueError("anomaly_types.csv contém IDs duplicados; revisar antes do merge.")

labels_rotuladas = labels_clean.merge(
    anomaly_types,
    on="ID",
    how="left",
    validate="many_to_one",
    indicator=True,
)

ids_sem_tipo = labels_rotuladas.loc[
    labels_rotuladas["_merge"] == "left_only", "ID"
].drop_duplicates()

resumo_auditoria = pd.DataFrame(
    {
        "verificacao": [
            "Linhas de labels no arquivo",
            "Linhas nos canais 41–46",
            "StartTime inválido no arquivo inteiro",
            "EndTime inválido no arquivo inteiro",
            "Linhas inválidas no recorte (data, ordem ou ID)",
            "IDs únicos no recorte com datas válidas",
            "IDs sem correspondência em anomaly_types.csv",
        ],
        "quantidade": [
            len(labels),
            len(labels_41_46_raw),
            labels_audit["StartTimeUTC"].isna().sum(),
            labels_audit["EndTimeUTC"].isna().sum(),
            len(labels_invalidas_41_46),
            labels_clean["ID"].nunique(),
            len(ids_sem_tipo),
        ],
    }
)

eventos_unicos = (
    labels_rotuladas.loc[labels_rotuladas["_merge"] == "both"]
    .groupby("ID", as_index=False)
    .agg(
        inicio_utc=("StartTimeUTC", "min"),
        fim_utc=("EndTimeUTC", "max"),
        canais_afetados=("Channel", "nunique"),
        n_intervalos_anotados=("Channel", "size"),
        Category=("Category", "first"),
        Class=("Class", "first"),
        Subclass=("Subclass", "first"),
    )
)
eventos_unicos["duracao_envelope"] = (
    eventos_unicos["fim_utc"] - eventos_unicos["inicio_utc"]
)

display(resumo_auditoria)
display(labels_invalidas_41_46[["ID", "Channel", "StartTime", "EndTime"]])
display(eventos_unicos.head())

,verificacao,quantidade
0,Linhas de labels no arquivo,3589
1,Linhas nos canais 41–46,1255
2,StartTime inválido no arquivo inteiro,1
3,EndTime inválido no arquivo inteiro,0
4,"Linhas inválidas no recorte (data, ordem ou ID)",0
5,IDs únicos no recorte com datas válidas,118
6,IDs sem correspondência em anomaly_types.csv,0


,ID,Channel,StartTime,EndTime


,ID,inicio_utc,fim_utc,canais_afetados,n_intervalos_anotados,Category,Class,Subclass,duracao_envelope
0,id_1,2004-12-02 03:07:17.646000+00:00,2004-12-08 23:35:55.146000+00:00,6,6,Rare Event,class_6,subclass_1,6 days 20:28:37.500000
1,id_10,2005-03-31 08:15:25.131000+00:00,2005-04-01 04:30:07.131000+00:00,6,6,Anomaly,class_9,subclass_1,0 days 20:14:42
2,id_100,2005-07-04 20:14:07.119000+00:00,2005-07-04 20:42:37.119000+00:00,6,12,Rare Event,class_14,subclass_1,0 days 00:28:30
3,id_101,2005-09-01 14:43:22.113000+00:00,2005-09-02 17:31:22.113000+00:00,6,6,Anomaly,class_20,unknown,1 days 02:48:00
4,id_102,2005-09-27 21:40:31.110000+00:00,2005-09-28 08:31:31.110000+00:00,6,6,Anomaly,class_20,unknown,0 days 10:51:00


In [52]:
contagem_eventos_categoria = (
    eventos_unicos["Category"]
    .value_counts(dropna=False)
    .rename_axis("Category")
    .reset_index(name="eventos_unicos")
)

display(contagem_eventos_categoria)
print(f"Total de eventos únicos válidos no recorte: {eventos_unicos['ID'].nunique()}")

,Category,eventos_unicos
0,Rare Event,63
1,Anomaly,51
2,Communication Gap,4


Total de eventos únicos válidos no recorte: 118


### Resultado da auditoria

| Verificação | Quantidade | O que significa |
|---|---|---|
| Linhas de labels no arquivo | 3.589 | Total de intervalos anotados, considerando todos os canais da missão. |
| Linhas nos canais 41–46 | 1.255 | Parte do arquivo que pertence ao recorte desta pesquisa. |
| `StartTime` inválido (arquivo inteiro) | 1 | Linha do `id_1` no `channel_15`, que contém o texto "Resultado" no lugar da data. Ela está fora dos canais 41–46 e não afeta esta análise. O valor não foi corrigido nem preenchido. |
| `EndTime` inválido (arquivo inteiro) | 0 | Nenhum problema encontrado. |
| Linhas inválidas no recorte | 0 | Nenhuma linha dos canais 41–46 foi descartada. Por isso, `labels_clean` tem as mesmas 1.255 linhas e `labels_invalidas_41_46` está vazia. |
| Eventos únicos com datas válidas | 118 | As 1.255 linhas correspondem a 118 eventos (cerca de 10 linhas por evento). |
| IDs sem correspondência em `anomaly_types.csv` | 0 | Todos os eventos têm categoria e classe definidas. |

### Eventos por categoria (canais 41–46)

| Categoria | Significado | Eventos |
|---|---|---|
| Rare Event | Evento nominal raro: comportamento incomum, mas esperado | 63 |
| Anomaly | Anomalia: comportamento incomum e indesejado | 51 |
| Communication Gap | Lacuna de comunicação: período sem dados | 4 |
| **Total** | | **118** |

### Pontos de atenção

- **O total de 118 eventos refere-se aos canais 41–46.** A missão completa (Mission1) tem 200 eventos anotados, dos quais 118 são anomalias. O número 118 aparece nas duas contagens, mas elas são diferentes e não devem ser confundidas.
- **A diferença entre 1.255 linhas e 118 eventos é esperada.** Cada evento é registrado em vários canais e, em alguns casos, em mais de um trecho.
- **As 4 lacunas de comunicação não são anomalias nem eventos raros.** Elas indicam ausência de dados. Nas análises seguintes, serão tratadas separadamente das outras duas categorias.

### Tabelas que serão usadas nas próximas seções

- `labels_clean`: uma linha por intervalo anotado em cada canal (1.255 linhas). É usada para recortar as séries de cada canal.
- `eventos_unicos`: uma linha por evento (118 linhas). É usada para contar eventos, comparar categorias e definir as janelas de tempo.

# Perguntas que iremos direcionar a análise:

#### 1 Entendendo como são os eventos considerados raros, anomalias e normais

**1.1 Quantos eventos existem nos canais 41–46? Desses, quantos são Anomaly e quantos são Rare Event?**

**1.2 Qual é a duração típica de uma anomalia? E de um evento nominal raro? Elas são mais longas ou mais curtas?**



**1.3 Existem eventos que afetam vários canais ao mesmo tempo (mesmo ID em múltiplos canais)? Com que frequência isso acontece?**

**1.4 Os eventos estão distribuídos uniformemente ao longo do tempo (2000–2013) ou existem períodos com mais ocorrências?**

#### 2. Comportamento dos canais durante eventos

**2.1 Como o valor do channel_41 se comporta durante uma anomalia anotada comparado com um período normal? O desvio é óbvio visualmente?**

**2.2 O mesmo evento aparece nos canais 41–46 ao mesmo tempo ou cada canal reage de forma diferente (em tempos diferentes)?**

**2.3 Existem períodos onde o sensor fica com o mesmo valor por muito tempo (valor "congelado")? Isso pode ser dado ruim, travamento ou sensor em modo de espera?**

**2.4 Qual é o intervalo típico entre leituras (frequência de amostragem)? Existem lacunas grandes - períodos sem nenhuma leitura?**

#### 3. Telecomandos e contexto operacinal

**3.1 Quantos telecomandos diferentes foram executados ao longo dos 14 anos? Quais são os mais frequentes?**

**3.2 Existe algum telecomando que aparece frequentemente nas horas ou dias que antecedem eventos nominais raros nos canais 41–46?**

**3.3 Esse mesmo telecomando também aparece antes de anomalias reais?**

**3.4 Qual é a prioridade dos telecomandos que aparecem próximos aos eventos? Comandos de prioridade 3 são mais associados a eventos do que comandos de prioridade 1?**

#### 4. Qualidade dos Dados

**4.1 Em que proporção do tempo os canais têm dados válidos? Existe algum canal com muito mais lacunas que os outros?**

**4.2 Durante as anomalias anotadas, os dados estavam completos ou havia lacunas/congelamentos que poderiam ter causado o alerta?**

**4.3 Existem valores extremos (muito acima ou abaixo do padrão histórico) que ocorrem fora de qualquer evento anotado? O que isso significa?**

#### 5. Detecção de Anomalias com Limites

**5.1 Se usarmos uma regra simples ("alerta quando o valor sai de 3 desvios padrões da mediana"), quantos alertas seriam gerados? Quantos coincidiriam com anomalias reais? Quantos seriam eventos nominais raros?**

**5.2 Um detector simples conseguiria distinguir os 118 eventos da Mission1 apenas pelo score numérico, sem contexto?**